In [ ]:
# Load abort event data
import subprocess
import sys

# Ensure openpyxl is installed
subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl", "-q"])

abort_data_path = "/Users/xylu/Desktop/Data/Database_Abort/Updated_LER_Event_Data_Analysis.xlsx"

print("="*60)
print("Loading Abort Event Data")
print("="*60)

if os.path.exists(abort_data_path):
    # Load abort data from Excel file
    abort_df = pd.read_excel(abort_data_path)
    
    # Parse the Time column (format: "M/D/YY H:MM" or "M/D/YY HH:MM")
    # Convert to datetime
    abort_df['abort_datetime'] = pd.to_datetime(abort_df['Time'], format='%m/%d/%y %H:%M')
    
    print(f"✓ Loaded {len(abort_df)} abort events")
    print(f"  Date range: {abort_df['abort_datetime'].min()} to {abort_df['abort_datetime'].max()}")
    print(f"  Columns: {list(abort_df.columns)}")
    
    # Filter abort events for the current date
    abort_date_start = datetime.strptime(date_str, "%Y%m%d").replace(hour=0, minute=0, second=0)
    abort_date_end = abort_date_start + timedelta(days=1)
    
    abort_mask = (abort_df['abort_datetime'] >= abort_date_start) & (abort_df['abort_datetime'] < abort_date_end)
    abort_today = abort_df[abort_mask].copy()
    
    print(f"\n✓ Found {len(abort_today)} abort event(s) on {date_str}")
    
    if len(abort_today) > 0:
        print("\nAbort Events:")
        for idx, row in abort_today.iterrows():
            print(f"  {row['abort_datetime'].strftime('%Y-%m-%d %H:%M:%S')} - {row['Category']} - {row['Origin']}")
    else:
        print("  No abort events found for this date")
else:
    print(f"⚠ Abort data file not found: {abort_data_path}")
    abort_df = None
    abort_today = None
    

In [ ]:
# User Input Parameters (CHANGE)
date_str = "20260302"  # YYYYMMDD format
start_time = "000000"  # HHMMSS format
end_time = "235959"  # HHMMSS format

# Optional: Set end_time_mask to limit x-axis display (set to None to disable)
# end_time_mask = "145200"  # HHMMSS format - cuts off display at this time
end_time_mask = end_time # for full-day plots

# Base path for acoustic data
base_path = "/Users/xylu/Desktop/Data/acoustic_vpp/"

# Convert strings to datetime objects for filtering
date_obj = datetime.strptime(date_str, "%Y%m%d")
start_time_obj = datetime.strptime(date_str + start_time, "%Y%m%d%H%M%S")
end_time_obj = datetime.strptime(date_str + end_time, "%Y%m%d%H%M%S")

# Convert mask time if provided
if end_time_mask is not None:
    end_time_mask_obj = datetime.strptime(date_str + end_time_mask, "%Y%m%d%H%M%S")
else:
    end_time_mask_obj = None

print(f"Date: {date_str}")
print(f"Time range: {start_time} - {end_time}")
print(f"Start: {start_time_obj.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"End: {end_time_obj.strftime('%Y-%m-%d %H:%M:%S')}")
if end_time_mask_obj:
    print(f"Display mask: {end_time_mask_obj.strftime('%Y-%m-%d %H:%M:%S')}")
    
# Load CSV files for the specified date
print("\n" + "="*60)
print("Looking for CSV files...")
print("="*60)

# Direct path: files are stored in /acoustic_vpp/YYYYMMDD/
date_folder = os.path.join(base_path, date_str)

if not os.path.exists(date_folder):
    print(f"⚠ Folder not found: {date_folder}")
    filtered_df = None
else:
    print(f"✓ Found folder: {date_folder}")
    
    # Find all CSV files matching the date pattern
    csv_files = sorted(glob.glob(os.path.join(date_folder, f"{date_str}*.csv")))
    print(f"  Found {len(csv_files)} CSV file(s)")
    
    if len(csv_files) == 0:
        print(f"⚠ No CSV files found matching {date_str}*.csv")
        filtered_df = None
    else:
        # Load and combine all CSV files
        all_data = []
        
        for csv_file in csv_files:
            try:
                df = pd.read_csv(csv_file)
                # Convert time_datetime to datetime if it's a string (handles microseconds)
                if 'time_datetime' in df.columns:
                    df['time_datetime'] = pd.to_datetime(df['time_datetime'], format='ISO8601')
                all_data.append(df)
                print(f"  ✓ {os.path.basename(csv_file)}: {len(df)} samples")
            except Exception as e:
                print(f"  ✗ Error reading {os.path.basename(csv_file)}: {e}")
        
        if all_data:
            # Combine all data
            combined_df = pd.concat(all_data, ignore_index=True)
            combined_df = combined_df.sort_values('time_datetime').reset_index(drop=True)
            
            print(f"\n✓ Combined data: {len(combined_df)} total samples")
            print(f"  Time range: {combined_df['time_datetime'].min()} to {combined_df['time_datetime'].max()}")
            
            # Filter by time range
            mask = (combined_df['time_datetime'] >= start_time_obj) & (combined_df['time_datetime'] <= end_time_obj)
            filtered_df = combined_df[mask].reset_index(drop=True)
            
            print(f"\n✓ Filtered data (between {start_time_obj.strftime('%H:%M:%S')} and {end_time_obj.strftime('%H:%M:%S')}): {len(filtered_df)} samples")
            
            if len(filtered_df) > 0:
                print(f"  Vpp range: {filtered_df['vpp_volts'].min():.4f} - {filtered_df['vpp_volts'].max():.4f} V")
            else:
                print("  ⚠ No data after filtering by time range")
        else:
            print("⚠ No data loaded!")
            filtered_df = None